# Module 6 - Deployment using Streamlit
Notebook ini adalah lanjutan dari notebook training. Sesuai anjuran modul, kode deployment dipisahkan ke file tersendiri. Setiap bagian aplikasi (langkah a sampai g) ditulis ke file `app.py` menggunakan magic command `%%writefile`. Cell pertama membuat file baru, sedangkan cell berikutnya memakai `%%writefile -a` untuk **menambahkan** kode di akhir file.

Deployment dilakukan secara lokal (localhost), sehingga tidak perlu membuat akun. Akun Streamlit Community Cloud baru diperlukan jika aplikasi ingin dipublikasikan secara online, misalnya untuk group project.

Library yang dibutuhkan:

```
pip install streamlit tensorflow scikit-learn joblib numpy pandas
```

Jalankan notebook ini pada folder yang sama dengan folder `model/` hasil notebook training.

### 4a. Load Model and Scaler
Bagian awal `app.py` mengimpor library dan memuat model serta scaler yang sudah disimpan. Modul tidak mencantumkan `import pandas as pd` padahal `pd` dipakai pada bagian berikutnya, sehingga ditambahkan di sini.

In [ ]:
%%writefile app.py
from tensorflow.keras.models import load_model
import joblib
import streamlit as st
import numpy as np
import pandas as pd

model = load_model("model/breast_cancer_model.keras")
scaler = joblib.load("model/scaler.pkl")

### 4b. Give a name for the application
`st.title` menampilkan judul aplikasi dan `st.write` menampilkan teks petunjuk singkat.

In [ ]:
%%writefile -a app.py
st.title("Breast Cancer Prediction App")

st.write(
    "Use either manual input or upload a CSV file "
    "to perform breast cancer prediction."
)

### 4c. Prepare features
`features` adalah daftar 30 nama fitur yang harus diisi pengguna, dengan urutan yang sama seperti saat model dilatih. Urutan ini penting karena scaler dan model menerima data berdasarkan posisi kolom.

Fungsi `normalize_column` adalah tambahan di luar modul. File `TEST-breast-cancer-pred.csv` memakai penamaan kolom gaya Kaggle (misalnya `radius_mean`, `radius_se`, `radius_worst`), sedangkan daftar fitur pada modul memakai penamaan scikit-learn (`mean radius`, `radius error`, `worst radius`). Tanpa penyesuaian, aplikasi akan selalu menampilkan pesan bahwa fitur CSV tidak lengkap. Fungsi ini mengubah nama kolom CSV ke format scikit-learn. Nama kolom yang sudah berformat scikit-learn tidak berubah, jadi kedua jenis CSV dapat digunakan.

In [ ]:
%%writefile -a app.py
features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean compactness",
    "mean concavity",
    "mean concave points",
    "mean symmetry",
    "mean fractal dimension",
    "radius error",
    "texture error",
    "perimeter error",
    "area error",
    "smoothness error",
    "compactness error",
    "concavity error",
    "concave points error",
    "symmetry error",
    "fractal dimension error",
    "worst radius",
    "worst texture",
    "worst perimeter",
    "worst area",
    "worst smoothness",
    "worst compactness",
    "worst concavity",
    "worst concave points",
    "worst symmetry",
    "worst fractal dimension"
]


def normalize_column(name):
    name = name.strip().replace("_", " ")
    if name.endswith(" mean"):
        return "mean " + name[:-5]
    if name.endswith(" se"):
        return name[:-3] + " error"
    if name.endswith(" worst"):
        return "worst " + name[:-6]
    return name

### 4d. Select input method
Aplikasi menyediakan dua cara input: **Manual Input** (pengguna mengisi nilai setiap fitur) dan **CSV File** (prediksi massal dari file CSV, dengan hasil prediksi untuk setiap baris).

In [ ]:
%%writefile -a app.py
input_method = st.radio(
    "Select Input Method:",
    ["Manual Input", "CSV File"],
    horizontal=True
)

### 4e. Manual Input
Form manual dibuat dalam dua kolom agar tampilan lebih ringkas. Fitur bernomor genap ditaruh di kolom kiri dan fitur bernomor ganjil di kolom kanan, dan setiap nilai yang diisi dikumpulkan dalam list `input_data`.

Satu tambahan kecil dari modul: parameter `format="%.5f"` pada `st.number_input`. Tanpa itu, Streamlit menampilkan dan membulatkan nilai ke dua angka desimal, padahal banyak fitur (misalnya *smoothness* dan *fractal dimension*) bernilai sekitar 0.05 hingga 0.1 sehingga pembulatan dapat memengaruhi hasil prediksi.

In [ ]:
%%writefile -a app.py
if input_method == "Manual Input":
    st.subheader("Enter Feature Values")
    input_data = []

    col1, col2 = st.columns(2)

    for i, feature in enumerate(features):
        if i % 2 == 0:
            with col1:
                value = st.number_input(
                    feature,
                    min_value=0.0,
                    value=0.0,
                    format="%.5f",
                    key=feature
                )
        else:
            with col2:
                value = st.number_input(
                    feature,
                    min_value=0.0,
                    value=0.0,
                    format="%.5f",
                    key=feature
                )
        input_data.append(value)

### 4f. CSV File Input
Jika pengguna memilih CSV, aplikasi menampilkan tombol upload. Setelah file dipilih, aplikasi menampilkan pratinjau data, memeriksa apakah seluruh 30 fitur yang dibutuhkan tersedia, lalu menampilkan pesan sukses beserta jumlah baris, atau pesan error beserta daftar fitur yang hilang.

In [ ]:
%%writefile -a app.py
else:
    st.subheader("Upload CSV File")
    uploaded_file = st.file_uploader(
        "Choose a CSV file",
        type=["csv"]
    )

    if uploaded_file is not None:
        raw_data = pd.read_csv(uploaded_file)
        csv_data = raw_data.rename(columns=normalize_column)
        st.write("Preview of uploaded data:")
        st.dataframe(raw_data.head())

        missing_features = [
            feature for feature in features
            if feature not in csv_data.columns
        ]

        if missing_features:
            st.error(
                "The CSV file is missing the following features:"
            )
            st.write(missing_features)

        else:
            st.success(
                f"CSV file loaded successfully. "
                f"{len(csv_data)} records found."
            )

### 4g. Run Prediction
Prediksi dijalankan ketika tombol **Run Predict** ditekan. Untuk manual input, nilai dibuat menjadi DataFrame satu baris, distandarisasi dengan scaler yang sama, lalu diprediksi oleh model. Untuk CSV, seluruh baris diprediksi sekaligus, hasilnya ditampilkan dalam tabel dengan ringkasan jumlah Benign dan Malignant, dan dapat diunduh sebagai file CSV.

**Koreksi label dari modul.** Pada modul, probabilitas `>= 0.5` dilabeli *Malignant*. Padahal pada dataset scikit-learn label 1 adalah *benign* (0 adalah *malignant*), dan keluaran sigmoid adalah probabilitas kelas 1. Artinya probabilitas `>= 0.5` seharusnya berarti **Benign**. Hal ini sudah dicek pada data: pada `TEST-breast-cancer-pred.csv`, kolom `diagnosis` bernilai `M` tepat pada baris-baris yang berlabel 0 di dataset scikit-learn. Karena itu di sini pemetaannya dibalik agar hasil prediksi tidak terbalik. Kolom probabilitas pada hasil CSV juga diberi nama `Benign Probability` supaya maknanya jelas, dan pada manual input ditampilkan kedua probabilitas (benign dan malignant).

In [ ]:
%%writefile -a app.py
st.divider()

if st.button("Run Predict", type="primary"):

    if input_method == "Manual Input":

        input_df = pd.DataFrame(
            [input_data], columns=features
        )

        input_scaled = scaler.transform(input_df.to_numpy())

        prediction = model.predict(
            input_scaled,
            verbose=0
        )

        probability_benign = float(prediction[0][0])

        if probability_benign >= 0.5:
            result = "Benign"
        else:
            result = "Malignant"

        st.subheader("Prediction Result")

        if result == "Malignant":
            st.error(f"Prediction: **{result}**")
        else:
            st.success(f"Prediction: **{result}**")

        st.write(
            f"Benign probability: **{probability_benign:.4f}**"
        )
        st.write(
            f"Malignant probability: **{1 - probability_benign:.4f}**"
        )

    else:
        if uploaded_file is None:
            st.warning(
                "Please upload a CSV file first."
            )
        elif missing_features:
            st.error(
                "Prediction cannot be performed because "
                "some required features are missing."
            )

        else:
            X_new = csv_data[features].to_numpy()

            X_scaled = scaler.transform(X_new)

            predictions = model.predict(X_scaled, verbose=0)

            probabilities = predictions.flatten()

            results = np.where(
                probabilities >= 0.5, "Benign", "Malignant"
            )

            result_df = raw_data.copy()
            result_df["Benign Probability"] = probabilities
            result_df["Prediction"] = results

            st.subheader("Prediction Results")
            st.dataframe(result_df)

            malignant_count = np.sum(results == "Malignant")
            benign_count = np.sum(results == "Benign")
            col1, col2 = st.columns(2)
            with col1:
                st.metric("Benign", benign_count)
            with col2:
                st.metric("Malignant", malignant_count)

            csv_output = result_df.to_csv(
                index=False
            )

            st.download_button(
                label="Download Prediction Results",
                data=csv_output,
                file_name="breast_cancer_predictions.csv",
                mime="text/csv"
            )

## 5. Run the model in Streamlit
Pastikan struktur folder proyek seperti berikut (`app.py` berada di luar folder `model/`):

```
breast-cancer-prediction/
│
├── model/
│   ├── breast_cancer_model.keras
│   └── scaler.pkl
│
└── app.py
```

**a. Buka Command Prompt / Terminal** pada folder `breast-cancer-prediction`, lalu instal library:

```
pip install streamlit tensorflow scikit-learn joblib numpy pandas
```

atau, jika sudah memiliki `requirements.txt`:

```
pip install -r requirements.txt
```

Jalankan Streamlit:

```
streamlit run app.py
```

Browser akan terbuka otomatis, biasanya di **http://localhost:8501**. Untuk uji prediksi CSV, gunakan file **TEST-breast-cancer-pred.csv**.

Catatan: menjalankan `streamlit run app.py` sebaiknya dilakukan dari terminal, bukan dari dalam cell notebook, karena perintah tersebut berjalan terus dan memblokir kernel.